# 04 · Updates, deletes and MERGE

**Time:** ~25 min · **You will learn:** row-level changes on plain files, and how to upsert with `MERGE INTO`.

Parquet files are immutable, so how can Iceberg `UPDATE` a row? By default it uses **copy-on-write**:
the affected data file is rewritten without (or with the changed) rows, and a new snapshot points to the new file.

In [ ]:
from helpers import get_spark, reset_orders

spark = get_spark("04-row-level")
reset_orders(spark, "orders", n=6, commits=1)
spark.table("demo.course.orders").orderBy("order_id").show()

## 1. UPDATE

In [ ]:
spark.sql("UPDATE demo.course.orders SET quantity = quantity + 10 WHERE order_id = 1001")
spark.sql("SELECT order_id, quantity FROM demo.course.orders WHERE order_id = 1001").show()

## 2. DELETE

In [ ]:
before = spark.table("demo.course.orders").count()
spark.sql("DELETE FROM demo.course.orders WHERE order_id = 1002")
print(before, "->", spark.table("demo.course.orders").count())

## 3. Each change is a snapshot
Look at the `operation` column: `append` for inserts, `overwrite` for update/delete that rewrote files.

In [ ]:
spark.sql("SELECT committed_at, operation, summary['added-records'] AS added, summary['deleted-records'] AS deleted FROM demo.course.orders.snapshots ORDER BY committed_at").show(truncate=False)

## 4. MERGE INTO (upsert)
`MERGE` compares a **source** (here: a DataFrame exposed as a temp view) with the **target** table and applies
different actions per case, in one atomic commit.

In [ ]:
from datetime import datetime
from helpers import ORDERS_DDL

changes = spark.createDataFrame([
    (1003, "carol", "monitor", 5, 179.0, datetime(2024, 1, 3, 8, 0)),   # existing id: change qty and price
    (2001, "erin",  "mouse",   1, 19.5,  datetime(2024, 1, 5, 9, 0)),   # new id: insert
], schema=ORDERS_DDL)
changes.createOrReplaceTempView("order_changes")

spark.sql("""
    MERGE INTO demo.course.orders AS t
    USING order_changes AS s
    ON t.order_id = s.order_id
    WHEN MATCHED THEN UPDATE SET t.quantity = s.quantity, t.unit_price = s.unit_price
    WHEN NOT MATCHED THEN INSERT *
""")
spark.table("demo.course.orders").orderBy("order_id").show()

You can also add `WHEN MATCHED AND s.quantity = 0 THEN DELETE` to remove rows. MERGE is the standard way to apply
change feeds (CDC) to Iceberg tables.

### Your turn

Write a `MERGE` that uses a source view `cancellations` containing order ids `1004` and `1005`
and **deletes** the matching orders. Assert that those two ids are gone and that the other rows are untouched.

In [ ]:
# Write your solution here


<details>
<summary>Show solution</summary>

```python
before = {r.order_id for r in spark.table("demo.course.orders").collect()}
spark.createDataFrame([(1004,), (1005,)], "order_id long").createOrReplaceTempView("cancellations")
spark.sql("""
    MERGE INTO demo.course.orders t
    USING cancellations c ON t.order_id = c.order_id
    WHEN MATCHED THEN DELETE
""")
after = {r.order_id for r in spark.table("demo.course.orders").collect()}
assert before - after == {1004, 1005}
assert after <= before
```

</details>

### Recap
* `UPDATE`, `DELETE` and `MERGE INTO` work on Iceberg tables and each commits a new snapshot.
* Default mode is copy-on-write; `write.update.mode=merge-on-read` writes small delete files instead (faster writes, slower reads until compaction).
* Old data is still there in earlier snapshots, which makes **time travel** possible.

Next: **05 · Time travel and rollback**.